# Mississippi 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Mississippi, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `grn_general_total`, `cst_general_total`, `ind_general_total`, `ref_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [3]:
# MS 2008 dataset path
PRIMARY_DEM_PATH  = r"../../data/raw/2008/MS/20080311__ms__democratic__primary.csv"
PRIMARY_REP_PATH  = r"../../data/raw/2008/MS/20080311__ms__republican__primary.csv"
GENERAL_PATH      = r"../../data/raw/2008/MS/20081104__ms__general__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/MS/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [4]:
# Load primary data
primary_dem = pd.read_csv(PRIMARY_DEM_PATH)
primary_rep = pd.read_csv(PRIMARY_REP_PATH)

# Join the two sub-dataframe into one
primary_df = pd.concat([primary_dem, primary_rep], ignore_index=True, sort=False)
primary_df.head(DISPLAY_ROWS)

,candidate,office,district,party,county,votes
0,Joseph Biden,President,NaN,Democrat,Adams,13
1,Joseph Biden,President,NaN,Democrat,Alcorn,31
2,Joseph Biden,President,NaN,Democrat,Amite,27
3,Joseph Biden,President,NaN,Democrat,Attala,24
4,Joseph Biden,President,NaN,Democrat,Benton,15
5,Joseph Biden,President,NaN,Democrat,Bolivar,20
6,Joseph Biden,President,NaN,Democrat,Calhoun,25
7,Joseph Biden,President,NaN,Democrat,Carroll,13
8,Joseph Biden,President,NaN,Democrat,Chickasaw,20
9,Joseph Biden,President,NaN,Democrat,Choctaw,9


In [5]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President      1476
U.S. House      505
U.S. Senate     164
Name: count, dtype: int64

In [6]:
# Only keep rows where 'office' is 'President'
primary_df = primary_df[primary_df["office"] == "President"]
primary_df.head(DISPLAY_ROWS)

,candidate,office,district,party,county,votes
0,Joseph Biden,President,NaN,Democrat,Adams,13
1,Joseph Biden,President,NaN,Democrat,Alcorn,31
2,Joseph Biden,President,NaN,Democrat,Amite,27
3,Joseph Biden,President,NaN,Democrat,Attala,24
4,Joseph Biden,President,NaN,Democrat,Benton,15
5,Joseph Biden,President,NaN,Democrat,Bolivar,20
6,Joseph Biden,President,NaN,Democrat,Calhoun,25
7,Joseph Biden,President,NaN,Democrat,Carroll,13
8,Joseph Biden,President,NaN,Democrat,Chickasaw,20
9,Joseph Biden,President,NaN,Democrat,Choctaw,9


In [7]:
# Primary data shape when only considering President
primary_df.shape

(1476, 6)

In [8]:
# Number of missing values in each column
primary_df.isna().sum()

candidate       0
office          0
district     1476
party           0
county          0
votes           0
dtype: int64

In [9]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,candidate,party,county,votes
0,Joseph Biden,Democrat,Adams,13
1,Joseph Biden,Democrat,Alcorn,31
2,Joseph Biden,Democrat,Amite,27
3,Joseph Biden,Democrat,Attala,24
4,Joseph Biden,Democrat,Benton,15
5,Joseph Biden,Democrat,Bolivar,20
6,Joseph Biden,Democrat,Calhoun,25
7,Joseph Biden,Democrat,Carroll,13
8,Joseph Biden,Democrat,Chickasaw,20
9,Joseph Biden,Democrat,Choctaw,9


In [10]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Joseph Biden              82
Hillary Rodham Clinton    82
Tom Tancredo              82
Mitt Romney               82
Alan Keyes                82
Duncan Hunter             82
Rudy Guiliani             82
Ron Paul                  82
John McCain               82
Mike Huckabee             82
Undecided                 82
Bill Richardson           82
Barack Obama              82
Dennis Kucinich           82
Mike Gravel               82
John Edwards              82
Chris Dodd                82
Fred Thompson             82
Name: count, dtype: int64

In [11]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
Democrat      738
Republican    738
Name: count, dtype: int64

In [12]:
# Data type of each column in primary_df
primary_df.dtypes

candidate    object
party        object
county       object
votes         int64
dtype: object

In [13]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,candidate,party,county,votes
0,Joseph Biden,Democrat,Adams,13
1,Joseph Biden,Democrat,Alcorn,31
2,Joseph Biden,Democrat,Amite,27
3,Joseph Biden,Democrat,Attala,24
4,Joseph Biden,Democrat,Benton,15
5,Joseph Biden,Democrat,Bolivar,20
6,Joseph Biden,Democrat,Calhoun,25
7,Joseph Biden,Democrat,Carroll,13
8,Joseph Biden,Democrat,Chickasaw,20
9,Joseph Biden,Democrat,Choctaw,9


In [14]:
# Shape after preprocessing
primary_df.shape

(1476, 4)

### b. General Election Dataset

In [15]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,candidate,party,votes
0,Adams,Courthouse,President,NaN,Barack Obama,Democrat,190
1,Adams,Bellemont,President,NaN,Barack Obama,Democrat,491
2,Adams,By-Pass Fire,President,NaN,Barack Obama,Democrat,678
3,Adams,Kingston,President,NaN,Barack Obama,Democrat,122
4,Adams,Liberty Park,President,NaN,Barack Obama,Democrat,135
5,Adams,Beau Pre,President,NaN,Barack Obama,Democrat,324
6,Adams,Duncan Park,President,NaN,Barack Obama,Democrat,337
7,Adams,Concord,President,NaN,Barack Obama,Democrat,651
8,Adams,Nps Multi Purpose Bldg.,President,NaN,Barack Obama,Democrat,266
9,Adams,Palestine,President,NaN,Barack Obama,Democrat,571


In [16]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President      12320
U.S. Senate     7040
U.S. House      4380
Name: count, dtype: int64

In [17]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,candidate,party,votes
0,Adams,Courthouse,President,NaN,Barack Obama,Democrat,190
1,Adams,Bellemont,President,NaN,Barack Obama,Democrat,491
2,Adams,By-Pass Fire,President,NaN,Barack Obama,Democrat,678
3,Adams,Kingston,President,NaN,Barack Obama,Democrat,122
4,Adams,Liberty Park,President,NaN,Barack Obama,Democrat,135
5,Adams,Beau Pre,President,NaN,Barack Obama,Democrat,324
6,Adams,Duncan Park,President,NaN,Barack Obama,Democrat,337
7,Adams,Concord,President,NaN,Barack Obama,Democrat,651
8,Adams,Nps Multi Purpose Bldg.,President,NaN,Barack Obama,Democrat,266
9,Adams,Palestine,President,NaN,Barack Obama,Democrat,571


In [18]:
# General data shape when only considering President
general_df.shape

(12320, 7)

In [19]:
# Number of missing values in each column
general_df.isna().sum()

county           0
precinct        14
office           0
district     12320
candidate        0
party            0
votes            0
dtype: int64

In [20]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,precinct,candidate,party,votes
0,Adams,Courthouse,Barack Obama,Democrat,190
1,Adams,Bellemont,Barack Obama,Democrat,491
2,Adams,By-Pass Fire,Barack Obama,Democrat,678
3,Adams,Kingston,Barack Obama,Democrat,122
4,Adams,Liberty Park,Barack Obama,Democrat,135
5,Adams,Beau Pre,Barack Obama,Democrat,324
6,Adams,Duncan Park,Barack Obama,Democrat,337
7,Adams,Concord,Barack Obama,Democrat,651
8,Adams,Nps Multi Purpose Bldg.,Barack Obama,Democrat,266
9,Adams,Palestine,Barack Obama,Democrat,571


Since the MS records are town-level, we’ll group by county and sum the votes to obtain county totals.

In [21]:
# Groupby county vote counts
general_df = (
    general_df.groupby(["county", "candidate", "party"], as_index=False)["votes"].sum()
)

general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Barack Obama,Democrat,18042
1,Adams,Bob Barr,Libertarian,94
2,Adams,Chuck Baldwin,Constitution,44
3,Adams,Cynthia A. McKinney,Green,12
4,Adams,John McCain,Republican,13132
5,Adams,Ralph Nader,Independent,60
6,Adams,Ted C. Weill / Frank McEnulty,Reform,8
7,Alcorn,Barack Obama,Democrat,8260
8,Alcorn,Bob Barr,Libertarian,90
9,Alcorn,Chuck Baldwin,Constitution,74


In [22]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Barack Obama                        72
Bob Barr                            72
Chuck Baldwin                       72
John McCain                         72
Ralph Nader                         72
Cynthia A. McKinney                 71
Ted C. Weill                        70
Ted C. Weill / Frank McEnulty        2
Cynthia McKinney / Rosa Clemente     1
Name: count, dtype: int64

There are three observations where there are two names in the `candidate` column. We need to have a closer look on those.

In [24]:
# Rows with 2 candidate names
general_df[(general_df["candidate"] == "Ted C. Weill / Frank McEnulty") | (general_df["candidate"] == "Cynthia McKinney / Rosa Clemente")]

,county,candidate,party,votes
6,Adams,Ted C. Weill / Frank McEnulty,Reform,8
10,Alcorn,Cynthia McKinney / Rosa Clemente,Green,36
13,Alcorn,Ted C. Weill / Frank McEnulty,Reform,14


As far as I understand, those are the presidential / vice-presidential tickets. Thus, we will keep only the presidential candidate, which is the first name.

In [25]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
)

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Barack Obama           72
Bob Barr               72
Chuck Baldwin          72
John McCain            72
Ralph Nader            72
Ted C. Weill           72
Cynthia A. McKinney    71
Cynthia McKinney        1
Name: count, dtype: int64

We also need to normalize name of "Cynthia McKinney" to "Cynthia A. McKinney" for consistency.

In [27]:
# Map the stray name to the canonical one
general_df["candidate"] = general_df["candidate"].replace({
    "Cynthia McKinney": "Cynthia A. McKinney"
})

# Updated candidate list in general_df
general_df["candidate"].value_counts()

candidate
Barack Obama           72
Bob Barr               72
Chuck Baldwin          72
Cynthia A. McKinney    72
John McCain            72
Ralph Nader            72
Ted C. Weill           72
Name: count, dtype: int64

In [28]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
Democrat        72
Libertarian     72
Constitution    72
Green           72
Republican      72
Independent     72
Reform          72
Name: count, dtype: int64

In [29]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [30]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Barack Obama,Democrat,18042
1,Adams,Bob Barr,Libertarian,94
2,Adams,Chuck Baldwin,Constitution,44
3,Adams,Cynthia A. McKinney,Green,12
4,Adams,John McCain,Republican,13132
5,Adams,Ralph Nader,Independent,60
6,Adams,Ted C. Weill,Reform,8
7,Alcorn,Barack Obama,Democrat,8260
8,Alcorn,Bob Barr,Libertarian,90
9,Alcorn,Chuck Baldwin,Constitution,74


In [31]:
# Shape after preprocessing
general_df.shape

(504, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [32]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democrat"     : "dem",
                "Republican"   : "rep",
                "Libertarian"  : "lib",
                "Green"        : "grn",
                "Constitution" : "cst",  
                "Independent"  : "ind",
                "Reform"       : "ref"
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [33]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [34]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [35]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNDECIDED,pri_rep_GUILIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_KEYES,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON
0,Adams,13,1539,9,48,10,21,4714,9,0,4,137,3,5,1028,111,30,2,9
1,Alcorn,31,3732,10,68,3,5,1061,15,0,2,176,5,13,910,50,15,1,9
2,Amite,27,930,13,31,13,6,1754,31,0,5,136,0,6,810,12,22,1,9
3,Attala,24,1393,12,43,7,8,2050,17,0,6,76,1,7,575,10,4,9,0
4,Benton,15,907,4,24,3,6,675,6,0,0,31,0,2,116,2,0,0,3
5,Bolivar,20,1421,11,27,8,22,5156,12,0,2,34,0,0,276,10,4,0,6
6,Calhoun,25,1447,4,36,3,11,1187,22,0,1,132,1,7,653,22,4,0,5
7,Carroll,13,892,2,12,5,5,1085,8,0,2,29,0,2,230,7,1,1,2
8,Chickasaw,20,1670,7,57,6,9,2235,16,0,3,89,1,4,541,14,6,0,11
9,Choctaw,9,636,3,32,4,5,780,11,0,3,92,1,4,455,11,3,0,7


Note that there is a column of undecided that still had votes (`pri_dem_UNDECIDED`). We will keep this for total counting purposes and drop it at the end.

In [36]:
# Primary dataframe shape after pivot
primary_pivot.shape

(82, 19)

In [37]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_ref_WEILL,gen_rep_MCCAIN
0,Adams,44,18042,12,60,94,8,13132
1,Alcorn,74,8260,36,280,90,14,21610
2,Amite,32,6696,16,38,18,10,8490
3,Attala,34,7698,12,34,36,6,10546
4,Benton,60,4454,12,60,28,4,4658
5,Bolivar,88,20668,32,56,106,30,9782
6,Calhoun,20,5044,12,32,18,8,8934
7,Chickasaw,42,9176,10,60,30,8,8790
8,Choctaw,14,2918,10,38,20,8,5248
9,Claiborne,16,9364,4,0,24,2,1496


In [38]:
# General dataframe shape after pivot
general_pivot.shape

(72, 8)

Here, for the pivoted general dataframe, we only have 72 rows. We will now figure out which counties are missing from the general dataset.

In [39]:
# List of all counties in `general_pivot`
general_pivot["county"].values

array(['Adams', 'Alcorn', 'Amite', 'Attala', 'Benton', 'Bolivar',
       'Calhoun', 'Chickasaw', 'Choctaw', 'Claiborne', 'Clarke', 'Clay',
       'Coahoma', 'Copiah', 'Desoto', 'Forrest', 'Franklin', 'George',
       'Hancock', 'Harrison', 'Hinds', 'Holmes', 'Humphreys', 'Issaquena',
       'Itawamba', 'Jackson', 'Jasper', 'Jefferson', 'Jefferson Davis',
       'Jones', 'Kemper', 'Lafayaette', 'Lamar', 'Lauderdale', 'Lawrence',
       'Leake', 'Lee', 'Leflore', 'Lincoln', 'Lowndes', 'Madison',
       'Marion', 'Marshall', 'Montgomery', 'Neshoba', 'Newton', 'Noxubee',
       'Pontotoc', 'Prentiss', 'Quitman', 'Rankin', 'Scott', 'Sharkey',
       'Simpson', 'Smith', 'Stone', 'Sunflower', 'Tallahatchie', 'Tate',
       'Tippah', 'Tishomingo', 'Tunica', 'Union', 'Walthall', 'Warren',
       'Washington', 'Wayne', 'Webster', 'Wilkinson', 'Winston',
       'Yalobusha', 'Yazoo'], dtype=object)

In [40]:
# Missing counties in general_pivot
missing_counties = [c for c in primary_pivot["county"].values if c not in general_pivot["county"].values]
missing_counties

['Carroll',
 'Covington',
 'DeSoto',
 'Greene',
 'Grenada',
 'Lafayette',
 'Monroe',
 'Oktibbeha',
 'Panola',
 'Pearl River',
 'Perry',
 'Pike']

## 4. Merge Dataframes

Given the missing counties in the general dataframe, the merged dataframe should likewise lack 17 counties, as shown in `general_pivot`.

In [41]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNDECIDED,...,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_ref_WEILL,gen_rep_MCCAIN
0,Adams,13,1539,9,48,10,21,4714,9,0,...,30,2,9,44,18042,12,60,94,8,13132
1,Alcorn,31,3732,10,68,3,5,1061,15,0,...,15,1,9,74,8260,36,280,90,14,21610
2,Amite,27,930,13,31,13,6,1754,31,0,...,22,1,9,32,6696,16,38,18,10,8490
3,Attala,24,1393,12,43,7,8,2050,17,0,...,4,9,0,34,7698,12,34,36,6,10546
4,Benton,15,907,4,24,3,6,675,6,0,...,0,0,3,60,4454,12,60,28,4,4658
5,Bolivar,20,1421,11,27,8,22,5156,12,0,...,4,0,6,88,20668,32,56,106,30,9782
6,Calhoun,25,1447,4,36,3,11,1187,22,0,...,4,0,5,20,5044,12,32,18,8,8934
7,Chickasaw,20,1670,7,57,6,9,2235,16,0,...,6,0,11,42,9176,10,60,30,8,8790
8,Choctaw,9,636,3,32,4,5,780,11,0,...,3,0,7,14,2918,10,38,20,8,5248
9,Claiborne,10,484,7,20,4,10,2262,17,0,...,0,2,0,16,9364,4,0,24,2,1496


In [42]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNDECIDED,pri_rep_GUILIANI,...,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_ref_WEILL,gen_rep_MCCAIN
count,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,...,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000
mean,22.114286,1916.614286,9.028571,48.514286,7.357143,11.028571,3332.971429,17.557143,0.600000,10.885714,...,26.500000,2.900000,25.271429,62.114286,13555.428571,25.200000,95.685714,58.542857,12.342857,16924.400000
std,14.003667,1777.767027,5.383742,36.487736,4.969576,9.040924,5371.931391,11.554077,2.799586,20.152709,...,56.659356,6.278904,54.805564,60.920367,18919.299087,26.314486,104.379261,64.352220,19.866136,18677.692663
min,2.000000,134.000000,1.000000,6.000000,0.000000,1.000000,297.000000,3.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.000000,1158.000000,2.000000,0.000000,2.000000,0.000000,728.000000
25%,14.000000,924.000000,6.000000,27.250000,4.000000,5.000000,1245.000000,11.000000,0.000000,2.000000,...,3.000000,0.000000,3.000000,24.000000,5987.500000,10.000000,32.000000,22.000000,4.000000,6229.000000
50%,19.500000,1385.500000,8.000000,38.500000,7.000000,9.500000,2057.500000,15.000000,0.000000,5.000000,...,7.500000,1.000000,9.000000,43.000000,8281.000000,16.000000,54.000000,34.000000,8.000000,10787.000000
75%,25.750000,2320.750000,11.000000,59.250000,9.000000,14.000000,3640.000000,19.000000,0.000000,9.000000,...,19.750000,2.000000,21.750000,73.000000,15502.500000,32.000000,131.000000,61.000000,12.000000,16874.000000
max,98.000000,9768.000000,28.000000,213.000000,33.000000,59.000000,43758.000000,62.000000,19.000000,128.000000,...,347.000000,40.000000,383.000000,312.000000,150802.000000,128.000000,524.000000,306.000000,128.000000,96280.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `grn_general_total` = sum of all `gen_grn_*` columns
    * `cst_general_total` = sum of all `gen_cst_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns
    * `ref_general_total` = sum of all `gen_ref_*` columns

In [43]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the UNDECIDED column.

In [45]:
# Drop UNCOMMITTED column for primary election
merged_df = merged_df.drop(columns="pri_dem_UNDECIDED")

# Snippet at the merged dataframe with primary totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GUILIANI,...,pri_rep_THOMPSON,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_ref_WEILL,gen_rep_MCCAIN,rep_primary_total,dem_primary_total
0,Adams,13,1539,9,48,10,21,4714,9,4,...,9,44,18042,12,60,94,8,13132,1329,6363
1,Alcorn,31,3732,10,68,3,5,1061,15,2,...,9,74,8260,36,280,90,14,21610,1181,4925
2,Amite,27,930,13,31,13,6,1754,31,5,...,9,32,6696,16,38,18,10,8490,1001,2805
3,Attala,24,1393,12,43,7,8,2050,17,6,...,0,34,7698,12,34,36,6,10546,688,3554
4,Benton,15,907,4,24,3,6,675,6,0,...,3,60,4454,12,60,28,4,4658,154,1640
5,Bolivar,20,1421,11,27,8,22,5156,12,2,...,6,88,20668,32,56,106,30,9782,332,6677
6,Calhoun,25,1447,4,36,3,11,1187,22,1,...,5,20,5044,12,32,18,8,8934,825,2735
7,Chickasaw,20,1670,7,57,6,9,2235,16,3,...,11,42,9176,10,60,30,8,8790,669,4020
8,Choctaw,9,636,3,32,4,5,780,11,3,...,7,14,2918,10,38,20,8,5248,576,1480
9,Claiborne,10,484,7,20,4,10,2262,17,0,...,0,16,9364,4,0,24,2,1496,44,2814


In [46]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
grn_general_cols   = [c for c in merged_df.columns if c.startswith("gen_grn_")]
cst_general_cols   = [c for c in merged_df.columns if c.startswith("gen_cst_")]
ind_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ind_")]
ref_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ref_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["grn_general_total"] = merged_df[grn_general_cols].sum(axis=1) if grn_general_cols else 0
merged_df["cst_general_total"] = merged_df[cst_general_cols].sum(axis=1) if cst_general_cols else 0
merged_df["ind_general_total"] = merged_df[ind_general_cols].sum(axis=1) if ind_general_cols else 0
merged_df["ref_general_total"] = merged_df[ref_general_cols].sum(axis=1) if ref_general_cols else 0

In [47]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_GRAVEL', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_rep_GUILIANI',
       'pri_rep_HUCKABEE', 'pri_rep_HUNTER', 'pri_rep_KEYES', 'pri_rep_MCCAIN',
       'pri_rep_PAUL', 'pri_rep_ROMNEY', 'pri_rep_TANCREDO',
       'pri_rep_THOMPSON', 'gen_cst_BALDWIN', 'gen_dem_OBAMA',
       'gen_grn_MCKINNEY', 'gen_ind_NADER', 'gen_lib_BARR', 'gen_ref_WEILL',
       'gen_rep_MCCAIN', 'rep_primary_total', 'dem_primary_total',
       'rep_general_total', 'dem_general_total', 'lib_general_total',
       'grn_general_total', 'cst_general_total', 'ind_general_total',
       'ref_general_total'],
      dtype='object')

In [48]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GUILIANI,...,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,lib_general_total,grn_general_total,cst_general_total,ind_general_total,ref_general_total
0,Adams,13,1539,9,48,10,21,4714,9,4,...,13132,1329,6363,13132,18042,94,12,44,60,8
1,Alcorn,31,3732,10,68,3,5,1061,15,2,...,21610,1181,4925,21610,8260,90,36,74,280,14
2,Amite,27,930,13,31,13,6,1754,31,5,...,8490,1001,2805,8490,6696,18,16,32,38,10
3,Attala,24,1393,12,43,7,8,2050,17,6,...,10546,688,3554,10546,7698,36,12,34,34,6
4,Benton,15,907,4,24,3,6,675,6,0,...,4658,154,1640,4658,4454,28,12,60,60,4
5,Bolivar,20,1421,11,27,8,22,5156,12,2,...,9782,332,6677,9782,20668,106,32,88,56,30
6,Calhoun,25,1447,4,36,3,11,1187,22,1,...,8934,825,2735,8934,5044,18,12,20,32,8
7,Chickasaw,20,1670,7,57,6,9,2235,16,3,...,8790,669,4020,8790,9176,30,10,42,60,8
8,Choctaw,9,636,3,32,4,5,780,11,3,...,5248,576,1480,5248,2918,20,10,14,38,8
9,Claiborne,10,484,7,20,4,10,2262,17,0,...,1496,44,2814,1496,9364,24,4,16,0,2


Now, we save the cleaned dataframe into the processed directory.

In [49]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "MS.csv", index=False)